# Day 2 - Week 4: Cross-Validation

### 1. Why Do We Need Cross-Validation?

In Day 1, we learned how to divide the dataset into:

Training set
Validation set
Test set

The validation set helps us evaluate the model while developing it.

However, there is one problem.

If we use only one validation split, the result may depend heavily on which rows happened to be placed in the validation set.

For example:

> Validation Split 1 → F1 = 0.81  
> Validation Split 2 → F1 = 0.73  
> Validation Split 3 → F1 = 0.78

Instead of trusting only one split, Cross-Validation evaluates the model several times using different validation data.

### 2. How k-Fold Cross-Validation Works

With 5-fold cross-validation, the training data is divided into 5 parts.

> Fold 1  
> Fold 2  
> Fold 3  
> Fold 4  
> Fold 5  

The model is trained 5 times.

| Round | Training Folds | Validation Fold |
|------:|----------------|-----------------:|
| 1     | 2, 3, 4, 5     | 1                |
| 2     | 1, 3, 4, 5     | 2                |
| 3     | 1, 2, 4, 5     | 3                |
| 4     | 1, 2, 3, 5     | 4                |
| 5     | 1, 2, 3, 4     | 5                |

This means every row:

is used for validation once
is used for training 4 times

At the end, we calculate the average score from all folds.

### 3. Imports

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression

import pandas as pd
import numpy as np

### 4. Load the Dataset

For this example, I will use the Breast Cancer classification dataset from Scikit-learn.

In [2]:
data = load_breast_cancer(as_frame=True)

df = data.frame

df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 31 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   mean radius              569 non-null    float64
 1   mean texture             569 non-null    float64
 2   mean perimeter           569 non-null    float64
 3   mean area                569 non-null    float64
 4   mean smoothness          569 non-null    float64
 5   mean compactness         569 non-null    float64
 6   mean concavity           569 non-null    float64
 7   mean concave points      569 non-null    float64
 8   mean symmetry            569 non-null    float64
 9   mean fractal dimension   569 non-null    float64
 10  radius error             569 non-null    float64
 11  texture error            569 non-null    float64
 12  perimeter error          569 non-null    float64
 13  area error               569 non-null    float64
 14  smoothness error         569 non-null

In [ ]:
X = df.drop("target", axis=1)
y = df["target"]
print("X shape:", X.shape)
print("y shape:", y.shape)
y.value_counts()

X shape: (569, 30)
y shape: (569,)


target
1    357
0    212
Name: count, dtype: int64

### 5. Create Train and Test Sets

We still keep a final test set separate.

The test set should not be used during cross-validation.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (455, 30)
X_test : (114, 30)
y_train: (455,)
y_test : (114,)


**Important**

Cross-validation will be performed only on:

X_train
y_train

The test set stays untouched until the final evaluation.

### 6. Create the Model

In [9]:
model = LogisticRegression(max_iter=5000)

### 7. Run 5-Fold Cross-Validation

We can use cross_val_score() to automatically train and validate the model several times.

In [11]:
scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=5,
    scoring="f1"
)
print(scores)

[0.97391304 0.95495495 0.94915254 0.96551724 0.96491228]


### 8. Mean Cross-Validation Score

Instead of looking at each fold separately, we calculate the average.

In [12]:
mean_score = scores.mean()

print("Mean F1 Score:", mean_score)

Mean F1 Score: 0.9616900125774326


The mean gives us our overall estimate of the model's performance.

### 9. Standard Deviation

We should also calculate the standard deviation.

In [13]:
std_score = scores.std()

print("Standard Deviation:", std_score)

Standard Deviation: 0.00868311392160267


means the model performs well and consistently.

### 10. Print the Full Cross-Validation Result

In [14]:
print("Scores:", scores)
print("Mean F1 Score:", scores.mean())
print("Standard Deviation:", scores.std())

Scores: [0.97391304 0.95495495 0.94915254 0.96551724 0.96491228]
Mean F1 Score: 0.9616900125774326
Standard Deviation: 0.00868311392160267


### Stratified K-Fold

For classification problems, we usually want every fold to contain approximately the same percentage of each class.

This is called Stratified K-Fold.

First, check the original class distribution:

In [15]:
y_train.value_counts(normalize=True)

target
1    0.626374
0    0.373626
Name: proportion, dtype: float64

Now create a `StratifiedKFold` object:

In [16]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

Use it with `cross_val_score`:

In [18]:
stratified_scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=skf,
    scoring="f1"
)
print("Scores:", stratified_scores)
print("Mean:", stratified_scores.mean())
print("Standard Deviation:", stratified_scores.std())

Scores: [0.97435897 0.93913043 0.95575221 0.96610169 0.94736842]
Mean: 0.9565423474997699
Standard Deviation: 0.012623575246896309


## Why Stratification Matters

Imagine our dataset contains:

> Class 0 → 20%  
> Class 1 → 80%  

A normal split could accidentally create a fold like:

> Class 0 → 5%  
> Class 1 → 95%  

Another fold could contain:

> Class 0 → 30%  
> Class 1 → 70%  

This makes the folds different from the original dataset.

Stratified K-Fold tries to preserve the original class proportions in every fold.

For classification problems, this gives us more representative validation set

### Inspect the Stratified Folds
We can see how the classes are distributed inside each validation fold.

In [19]:
for fold, (train_index, validation_index) in enumerate(
    skf.split(X_train, y_train),
    start=1
):
    y_validation_fold = y_train.iloc[validation_index]

    print(f"Fold {fold}")
    print(y_validation_fold.value_counts(normalize=True))
    print()

Fold 1
target
1    0.626374
0    0.373626
Name: proportion, dtype: float64

Fold 2
target
1    0.626374
0    0.373626
Name: proportion, dtype: float64

Fold 3
target
1    0.626374
0    0.373626
Name: proportion, dtype: float64

Fold 4
target
1    0.626374
0    0.373626
Name: proportion, dtype: float64

Fold 5
target
1    0.626374
0    0.373626
Name: proportion, dtype: float64



Each fold has almost the same class proportions. Class 1 represents about 62.6% and Class 0 about 37.4% in every fold. This shows that Stratified K-Fold preserves the original class distribution across the validation folds.